# Superconducting Resonator — Time-Dependent S₂₁ Pipeline

End-to-end pipeline that takes **raw VNA sweep CSVs** and produces **fitted
resonator parameters (Qᵢ, Qc, φ, f꜀) and their noise power spectral densities**,
using the DCM circle-fit routines from
[WeiRenSyong/Fitting_Code_Lab_2.0](https://github.com/WeiRenSyong/Fitting_Code_Lab_2.0).

## Pipeline overview

```
 raw VNA CSVs (flat folder)
        │
        │  STEP 1 · Organize  — parse filenames, group by power, re-index
        ▼
 organized/   + metadata.csv
        │
        │  STEP 2 · Trim      — inspect dip-tracking plot, discard unstable
        ▼                       early sweeps
 trimmed/     + time_sweep_summary.csv   ◄── THE DATA CONTRACT (see Section 2)
        │
        │  STEP 3 · Fit       — DCM circle fit on every sweep
        ▼
 fitted parameters per power
        │
        │  STEP 4 · Analyze   — time-series plots + Welch PSDs
        ▼
 PNG figures + fitted_params/*.csv
```

## How the two halves stay consistent

Previously the data-handling code wrote `metadata_trimmed.csv` (flat folder,
power in column 0, unix time in column 4) while the analysis code looked for
`time_sweep_summary.csv` with timestamps at column index 3 and power at index 5,
inside per-power *subfolders*. None of those assumptions matched, so the
analysis either crashed or silently mis-parsed integer indices as 1970-era
timestamps.

This notebook fixes that with a single **data contract** (Section 2): the trim
step *writes* `time_sweep_summary.csv` with explicitly named columns, and the
analysis *reads those same named columns* — no hard-coded column indices, no
folder-name guessing. If you only need to re-run the analysis on already-trimmed
data, you can restart the kernel and run Sections 0–2, then jump straight to
Section 5.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scipy`, plus the packages
required by Fitting_Code_Lab_2.0 (`lmfit`, `uncertainties`, `attrs`, `inflect`,
`regex`).

## Section 0 — Configuration (edit this cell only)

Every user-editable setting lives here. Nothing below this cell should need
editing for a normal run.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  PATHS
# ════════════════════════════════════════════════════════════════════
from pathlib import Path

# Flat folder of raw VNA sweep CSVs produced by the measurement script.
SOURCE_DIR = Path(r"C:\Users\user\Downloads\15mK_Time_Dependent_S21\15mK_Time_Dependent_S21\06_10_174033")

# Where the organized / trimmed data and all outputs will be written.
TARGET_DIR = Path(r"C:\Users\user\OneDrive\文件\GitHub\Measurements\Cooldown_78_Line1-HTCav_01_CPW_w50g25_03")

# Label for this resonator (used in output folder names).
RES_LABEL = "Resonator_1"

# Local clone of the fitting repository (root of Fitting_Code_Lab_2.0).
REPO_PATH = Path(r"C:\Users\user\OneDrive\文件\GitHub\Fitting_Code_Lab_2.0")

# ════════════════════════════════════════════════════════════════════
#  TRIMMING
# ════════════════════════════════════════════════════════════════════
# Sweeps with file index < TRIM_START_INDEX are discarded (the cryostat
# is usually still thermalizing when the measurement starts).
# Run Section 4's dip-tracking plot first, then set this value.
TRIM_START_INDEX = 0          # ◄── UPDATE after inspecting the dip plot!

# ════════════════════════════════════════════════════════════════════
#  FITTING (DCM Monte-Carlo settings, passed to the repository)
# ════════════════════════════════════════════════════════════════════
MC_ITERATION  = 10
MC_ROUNDS     = 10
MC_STEP_CONST = 0.3

# ════════════════════════════════════════════════════════════════════
#  OUTPUT TOGGLES
# ════════════════════════════════════════════════════════════════════
SAVE_PLOTS        = True   # summary PNG figures next to the data
SAVE_CSVS         = True   # fitted parameters -> fitted_params/*.csv
SAVE_CIRCLE_PLOTS = True   # one circle-fit PNG per sweep (repo format)

print("Configuration loaded.")
print(f"  Source : {SOURCE_DIR}")
print(f"  Target : {TARGET_DIR}")
print(f"  Repo   : {REPO_PATH}")

## Section 1 — Imports & fitting-repository setup

All imports for the whole pipeline live here (previously they were duplicated
across two separate "halves" of the notebook). We also add the repository's
`scresonators` package to `sys.path` so we can call `fit_resonator.fit.fit()`
and `fit_resonator.resonator.Resonator()` directly — the exact same routines
used for power-sweep analysis.

In [ ]:
import logging
import re
import shutil
import sys
import tempfile
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sig

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print("Standard imports successful.")

In [ ]:
# ── Make the fitting repository importable ───────────────────────────
repo_path = REPO_PATH.resolve()
if not repo_path.is_dir():
    raise FileNotFoundError(
        f"Repository not found: {repo_path}\n"
        "Please update REPO_PATH in the Configuration cell (Section 0)."
    )

scresonators_path = repo_path / "scresonators"
if not scresonators_path.is_dir():
    raise FileNotFoundError(
        f"'scresonators' subfolder not found inside {repo_path}\n"
        "Check that REPO_PATH points to the root of Fitting_Code_Lab_2.0."
    )

# Insert at the front so the repo's modules take priority over anything
# else with the same name that might be installed.
for p in (scresonators_path, repo_path / "helper_scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import fit_resonator.fit       as fsd
import fit_resonator.resonator as res

print(f"fit_resonator.fit      : {fsd.__file__}")
print(f"fit_resonator.resonator: {res.__file__}")
print("Repository loaded successfully.")

## Section 2 — The data contract

This is the **single source of truth** that keeps the data-handling and
analysis halves consistent. The trim step (Section 4) *writes*
`time_sweep_summary.csv` using these column names, and the analysis
(Section 5) *reads* the same names back.

| column             | meaning                                                       |
|--------------------|---------------------------------------------------------------|
| `power_str`        | input power label, e.g. `-20dBm` (groups the sweeps)          |
| `file_index`       | sweep index *after* trimming (0, 1, 2, … per power)           |
| `orig_file_index`  | sweep index in the organized (pre-trim) data                  |
| `source_index`     | index parsed from the original raw filename                   |
| `unix_time`        | file modification time (s since epoch) ≈ end of that sweep    |
| `time_s`           | seconds elapsed since the first kept sweep of the same power  |
| `filename`         | CSV filename, relative to the trimmed folder                  |

Because the analysis looks up columns **by name**, reordering or adding
columns can never silently break it (unlike the old hard-coded
column-index approach, which read an integer counter as a timestamp).

In [ ]:
# ── The contract: file name + column names used by writer AND reader ─
SUMMARY_FILENAME = "time_sweep_summary.csv"

COL_POWER        = "power_str"
COL_FILE_INDEX   = "file_index"
COL_ORIG_INDEX   = "orig_file_index"
COL_SOURCE_INDEX = "source_index"
COL_UNIX_TIME    = "unix_time"
COL_TIME_S       = "time_s"
COL_FILENAME     = "filename"

SUMMARY_COLUMNS = [
    COL_POWER, COL_FILE_INDEX, COL_ORIG_INDEX, COL_SOURCE_INDEX,
    COL_UNIX_TIME, COL_TIME_S, COL_FILENAME,
]

# Pretty axis labels for the fitted parameters (used in Section 7).
PARAM_LABELS = {
    "Qi":  r"$Q_i$",
    "Qc":  r"$Q_c$",
    "phi": r"$\phi$ (rad)",
    "fc":  r"$f_c$ (GHz)",
}

print("Data contract defined:", ", ".join(SUMMARY_COLUMNS))

## Section 3 — Step 1 · Organize the raw measurement files

**Input** — a flat source directory of VNA sweep CSVs, e.g.

```
QSD_CPW_w6g3_03_5p608GHz_-20dBm_15mK_0.csv        ← 3-column sweep (used)
QSD_CPW_w6g3_03_5p608GHz_-20dBm_15mK_0_full.csv   ← 13-column details (skipped)
QSD_CPW_w6g3_03_5p608GHz_-40dBm_15mK_0.csv
...
```

The 3-column files already have the format the fitting code needs
(`Frequency Hz, S21 magn_dB, S21 phase_deg`, with a header row).

**Output** — `{TARGET_DIR}/{RES_LABEL}_{freq_str}/` containing the same files,
re-indexed from 0 within each power group, plus a `metadata.csv` bookkeeping
file.

### 3.1 Filename parser

The expected filename stem is `{sample}_{freq}_{power}_{temperature}_{index}`.
`_full` files are excluded naturally: their stem ends in `_full`, not a digit,
so the regex simply does not match.

In [ ]:
# Regex anchored on the known-format suffix; the sample name floats.
#   Example match: QSD_CPW_w6g3_03_5p608GHz_-20dBm_15mK_0
_FILE_RE = re.compile(r'^(.+?)_([\dp]+GHz)_([-\dp]+dBm)_(\w+?)_(\d+)$')


def parse_sweep_file(stem: str) -> Optional[dict]:
    """Parse a sweep CSV stem into its parts.

    Returns None for non-matching names (e.g. `..._full` detail files),
    which is how those files get skipped.
    """
    m = _FILE_RE.match(stem)
    if not m:
        return None
    sample, freq_str, power_str, temperature, idx = m.groups()
    return {
        "sample_name": sample,
        "freq_str":    freq_str,
        "power_str":   power_str,
        "temperature": temperature,
        "file_index":  int(idx),
    }


# Quick self-test so a format change is caught immediately.
_test_cases = [
    ("QSD_CPW_w6g3_03_5p608GHz_-20dBm_15mK_0",      True),
    ("QSD_CPW_w6g3_03_5p608GHz_-20dBm_15mK_0_full", False),
]
for stem, expect_match in _test_cases:
    result = parse_sweep_file(stem)
    ok = (result is not None) == expect_match
    print(f"  {'OK  ' if ok else 'FAIL'}  '{stem}' -> {result}")

### 3.2 Scan the source directory and group sweeps by power

In [ ]:
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

sweep_entries = []   # one dict per matched sweep CSV
skipped       = []   # filenames that did not match (e.g. _full files)

for csv_path in sorted(SOURCE_DIR.glob("*.csv")):
    info = parse_sweep_file(csv_path.stem)
    if info is None:
        skipped.append(csv_path.name)
        continue
    info["csv_path"] = csv_path
    # File modification time ≈ when the measurement script finished
    # writing this sweep. This becomes the sweep's timestamp downstream.
    info["mtime"] = csv_path.stat().st_mtime
    sweep_entries.append(info)

if not sweep_entries:
    raise RuntimeError(f"No sweep CSVs matched the expected filename "
                       f"format in {SOURCE_DIR}")

print(f"Matched sweep CSVs  : {len(sweep_entries)}")
print(f"Skipped (e.g. _full): {len(skipped)}")

# ── Keep only the dominant center frequency (warn if several found) ──
from collections import Counter
freq_counts = Counter(e["freq_str"] for e in sweep_entries)
if len(freq_counts) > 1:
    print(f"[WARNING] Multiple frequencies found: {dict(freq_counts)}")
freq_str = freq_counts.most_common(1)[0][0]
sweep_entries = [e for e in sweep_entries if e["freq_str"] == freq_str]
print(f"Center frequency    : {freq_str}")

sample_name = sweep_entries[0]["sample_name"]
temperature = sweep_entries[0]["temperature"]
print(f"Sample name         : {sample_name}")
print(f"Temperature         : {temperature}")

# ── Group by power, sort each group by its original file index ───────
power_groups: Dict[str, list] = defaultdict(list)
for e in sweep_entries:
    power_groups[e["power_str"]].append(e)

all_powers = sorted(power_groups.keys(),
                    key=lambda p: float(p.replace("dBm", "").replace("p", ".")))

print(f"\nPowers found: {all_powers}")
for pwr in all_powers:
    power_groups[pwr].sort(key=lambda e: e["file_index"])
    g = power_groups[pwr]
    print(f"  {pwr:>10s} : {len(g):>5d} sweeps  "
          f"(index {g[0]['file_index']:04d} -> {g[-1]['file_index']:04d})")

### 3.3 Copy into the organized folder and write `metadata.csv`

Files are re-indexed from 0 within each power group so the sequence is gapless
even if some raw indices were missing.

In [ ]:
organized_dir = TARGET_DIR / f"{RES_LABEL}_{freq_str}"
organized_dir.mkdir(parents=True, exist_ok=True)

metadata_rows = []
for pwr in all_powers:
    for new_idx, entry in enumerate(power_groups[pwr]):
        new_name = f"{sample_name}_{freq_str}_{pwr}_{temperature}_{new_idx}.csv"
        shutil.copy2(entry["csv_path"], organized_dir / new_name)
        metadata_rows.append({
            COL_POWER:        pwr,
            COL_FILE_INDEX:   new_idx,
            COL_SOURCE_INDEX: entry["file_index"],
            COL_UNIX_TIME:    entry["mtime"],
            COL_FILENAME:     new_name,
        })

meta_df = pd.DataFrame(metadata_rows)
meta_df.to_csv(organized_dir / "metadata.csv", index=False)

print(f"Organized files saved to: {organized_dir}")
print(f"Total files copied      : {len(meta_df)}")
print()
print(meta_df.groupby(COL_POWER)[COL_FILE_INDEX].count()
      .rename("n_sweeps").to_string())

In [ ]:
# ── Sanity check: estimate the sweep cadence from the timestamps ─────
# Median Δt between consecutive files of the same power. If the files
# were copied (not freshly measured), mtimes may be wrong — this print
# lets you catch that before trusting any time axis downstream.
dt_samples = []
for pwr in all_powers:
    mtimes = np.array([e["mtime"] for e in power_groups[pwr]])
    if len(mtimes) >= 2:
        dt_samples.extend(np.diff(mtimes).tolist())

if dt_samples:
    sweep_duration = float(np.median(dt_samples))
    print(f"Estimated sweep duration : {sweep_duration:.3f} s "
          f"({sweep_duration/60:.2f} min)")
    print(f"Sampling frequency (1/dt): {1/sweep_duration:.4f} Hz")
    print()
    print("If this looks wrong (e.g. files were copied after measurement,")
    print("so mtimes are all identical), the time axes and PSDs below")
    print("cannot be trusted — re-copy with original timestamps preserved.")
else:
    print("Not enough files to estimate sweep duration.")

## Section 4 — Step 2 · Trim the unstable start of the measurement

When a measurement starts before the cryostat temperature has fully
stabilized, the first sweeps drift. We track the **resonance dip** of every
sweep (cheap — no fitting needed) and plot it so you can see where things
settle:

* **dip frequency** — sensitive to temperature drift,
* **dip |S₂₁|** — sensitive to coupling / loss changes.

Then set `TRIM_START_INDEX` in Section 0 (or override below) and run the trim
cell, which writes the trimmed data **and** the `time_sweep_summary.csv`
contract file consumed by the analysis.

### 4.1 Track the resonance dip of every sweep

In [ ]:
dip_data = {}   # power -> dict of arrays (sweep_indices, dip freq, dip mag)

for pwr in all_powers:
    n = len(power_groups[pwr])
    dip_freqs_GHz = np.full(n, np.nan)
    dip_mags_dB   = np.full(n, np.nan)

    for file_idx in range(n):
        fname = f"{sample_name}_{freq_str}_{pwr}_{temperature}_{file_idx}.csv"
        data = np.loadtxt(organized_dir / fname, delimiter=",", skiprows=1)
        freqs_Hz = data[:, 0]
        magn_dB  = data[:, 1]
        dip_idx  = np.argmin(magn_dB)          # deepest point of the dip
        dip_freqs_GHz[file_idx] = freqs_Hz[dip_idx] / 1e9
        dip_mags_dB[file_idx]   = magn_dB[dip_idx]

    dip_data[pwr] = {
        "sweep_indices": np.arange(n, dtype=int),
        "dip_freqs_GHz": dip_freqs_GHz,
        "dip_mags_dB":   dip_mags_dB,
    }
    print(f"  {pwr}: dip tracking done ({n} sweeps)")

print("Done.")

In [ ]:
# ── Plot dip frequency (top) and dip |S21| (bottom) vs sweep index ───
n_pwr = len(all_powers)
fig, axes = plt.subplots(2, n_pwr,
                         figsize=(max(5 * n_pwr, 8), 6),
                         sharex="col", sharey="row")
axes = np.atleast_2d(axes).reshape(2, n_pwr)   # 2-D even for one power

for col, pwr in enumerate(all_powers):
    d = dip_data[pwr]
    axes[0, col].plot(d["sweep_indices"], d["dip_freqs_GHz"],
                      ".", ms=2, color="C0")
    axes[1, col].plot(d["sweep_indices"], d["dip_mags_dB"],
                      ".", ms=2, color="C1")
    axes[0, col].set_title(pwr, fontsize=11)
    axes[1, col].set_xlabel("Sweep index (file_index)")

axes[0, 0].set_ylabel("Dip frequency (GHz)")
axes[1, 0].set_ylabel("Dip |S21| (dB)")

fig.suptitle(
    f"{sample_name}  |  {RES_LABEL}_{freq_str}  |  {temperature}\n"
    "Resonance dip tracking — identify the stable-temperature onset",
    fontsize=11,
)
fig.tight_layout()
plt.show()

print("\n-> Inspect the plot, then set TRIM_START_INDEX "
      "(Section 0, or override in the next cell).")

### 4.2 Apply the trim and write the data contract

This is the cell that previously broke the pipeline: it wrote
`metadata_trimmed.csv` with a column order the analysis didn't expect.
It now writes **`time_sweep_summary.csv`** using the named columns from
Section 2, into the trimmed folder. The analysis half reads exactly this
file — by column name.

In [ ]:
# Optional override (otherwise the value from Section 0 is used):
# TRIM_START_INDEX = 120

print(f"TRIM_START_INDEX = {TRIM_START_INDEX}")
for pwr in all_powers:
    n_total = len(power_groups[pwr])
    n_kept  = max(n_total - TRIM_START_INDEX, 0)
    print(f"  {pwr:>10s} : keep {n_kept}/{n_total}  "
          f"(discard first {min(TRIM_START_INDEX, n_total)} sweeps)")

In [ ]:
trimmed_dir = TARGET_DIR / f"{RES_LABEL}_{freq_str}_trimmed"
trimmed_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for pwr in all_powers:
    group = power_groups[pwr]

    # Keep only sweeps at/after the trim point, re-index from 0.
    kept = [(orig_idx, entry) for orig_idx, entry in enumerate(group)
            if orig_idx >= TRIM_START_INDEX]
    if not kept:
        log.warning("Power %s: trim removed every sweep — skipping.", pwr)
        continue

    t0 = kept[0][1]["mtime"]   # timestamp of the first kept sweep

    for new_idx, (orig_idx, entry) in enumerate(kept):
        src_name = f"{sample_name}_{freq_str}_{pwr}_{temperature}_{orig_idx}.csv"
        dst_name = f"{sample_name}_{freq_str}_{pwr}_{temperature}_{new_idx}.csv"
        shutil.copy2(organized_dir / src_name, trimmed_dir / dst_name)

        summary_rows.append({
            COL_POWER:        pwr,
            COL_FILE_INDEX:   new_idx,
            COL_ORIG_INDEX:   orig_idx,
            COL_SOURCE_INDEX: entry["file_index"],
            COL_UNIX_TIME:    entry["mtime"],
            COL_TIME_S:       entry["mtime"] - t0,   # measured, not assumed
            COL_FILENAME:     dst_name,
        })

summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_path = trimmed_dir / SUMMARY_FILENAME
summary_df.to_csv(summary_path, index=False)

print(f"Trimmed files saved to : {trimmed_dir}")
print(f"Summary (contract) file: {summary_path}")
print()
print(summary_df.groupby(COL_POWER)[COL_FILE_INDEX].count()
      .rename("n_sweeps_kept").to_string())

## Section 5 — Step 3 · Load the summary and map sweeps to files

**Standalone entry point.** If your data is already organized and trimmed,
you can restart the kernel, run Sections 0–2 (config, imports, contract), and
then start here — this section reads everything it needs from disk.

`load_time_sweep_summary()` validates the contract columns by **name** and
returns a `power_map`:

```
power_map = { "-20dBm": [(timestamp, csv_path), ...],   # chronological
              "-40dBm": [...], ... }
```

In [ ]:
def load_time_sweep_summary(folder: Path) -> Dict[str, List[Tuple]]:
    """Read the contract file and map each sweep to its CSV on disk.

    Returns
    -------
    dict : power_str -> list of (timestamp, csv_path), sorted by time.

    Raises a clear error if the file or any contract column is missing,
    or if a referenced CSV does not exist.
    """
    summary_path = folder / SUMMARY_FILENAME
    if not summary_path.exists():
        raise FileNotFoundError(
            f"'{SUMMARY_FILENAME}' not found in:\n  {folder}\n"
            "Run the data-handling steps (Sections 3-4) first, or point "
            "this section at a folder produced by them."
        )

    df = pd.read_csv(summary_path)

    # ── Validate the contract by column NAME (not position) ──────────
    missing = [c for c in (COL_POWER, COL_UNIX_TIME, COL_FILENAME)
               if c not in df.columns]
    if missing:
        raise ValueError(
            f"{SUMMARY_FILENAME} is missing required column(s): {missing}\n"
            f"Columns present: {list(df.columns)}\n"
            "This file must be written by the trim step (Section 4.2)."
        )

    log.info("Summary file: %s  (%d rows)", summary_path.name, len(df))

    # unix_time (seconds since epoch) -> timezone-aware timestamps
    df["timestamp"] = pd.to_datetime(df[COL_UNIX_TIME], unit="s", utc=True)

    power_map: Dict[str, List[Tuple]] = {}
    missing_files = []
    for power, grp in df.groupby(COL_POWER):
        grp = grp.sort_values("timestamp")
        entries = []
        for _, row in grp.iterrows():
            csv_path = folder / row[COL_FILENAME]
            if not csv_path.exists():
                missing_files.append(row[COL_FILENAME])
                continue
            entries.append((row["timestamp"], csv_path))
        if entries:
            power_map[power] = entries

    if missing_files:
        log.warning("%d file(s) listed in the summary were not found on "
                    "disk and were skipped: %s ...",
                    len(missing_files), missing_files[:3])
    if not power_map:
        raise RuntimeError("No usable sweeps found — every file listed in "
                           f"{SUMMARY_FILENAME} is missing from {folder}.")

    log.info("Found %d power level(s): %s",
             len(power_map), sorted(power_map))
    return power_map

In [ ]:
# The analysis folder defaults to the trimmed output of Section 4.
# Running standalone? Point ANALYSIS_DIR at any folder that contains
# time_sweep_summary.csv plus the sweep CSVs it lists.
try:
    ANALYSIS_DIR = trimmed_dir            # defined if Sections 3-4 ran
except NameError:
    ANALYSIS_DIR = TARGET_DIR / "Resonator_1_5p608GHz_trimmed"  # <- edit me

ANALYSIS_DIR = Path(ANALYSIS_DIR).resolve()
print(f"Analysis folder: {ANALYSIS_DIR}")

power_map = load_time_sweep_summary(ANALYSIS_DIR)
for pwr, entries in sorted(power_map.items()):
    print(f"  {pwr:>10s} : {len(entries)} sweeps, "
          f"{entries[0][0]:%Y-%m-%d %H:%M:%S} -> {entries[-1][0]:%H:%M:%S}")

## Section 6 — Step 4a · DCM circle fit on every sweep

`fit_one_file()` calls `fit_resonator.fit.fit()` — the **same function** used
by `fit_single_res()` in `helper_fit.py` for power-sweep analysis. Because the
repository saves its PNG relative to the CSV's location, each CSV is first
copied to a temporary folder; the fit runs there, and the output PNG is then
moved back next to the original CSV.

In [ ]:
def fit_one_file(csv_path: Path, save_png: bool = True) -> Optional[Dict]:
    """Fit a single S21 CSV with the repository's DCM routine.

    Returns a dict with Qi, Qc, phi, fc (GHz) and their 1-sigma errors,
    or None if the fit fails / gives an unphysical Qi.
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_csv = Path(tmpdir) / csv_path.name
        shutil.copy2(csv_path, tmp_csv)

        # Configure the resonator object exactly like the power-sweep flow.
        myres = res.Resonator()
        myres.from_file(str(tmp_csv))
        myres.preprocess_method = "circle"
        myres.normalize         = 5
        myres.save_dcm_plot     = save_png
        myres.plot_extra        = False
        myres.plot              = "png"
        myres.fit_dir           = tmpdir

        myres.fit_method(
            "DCM",
            MC_iteration  = MC_ITERATION,
            MC_rounds     = MC_ROUNDS,
            MC_fix        = [],
            manual_init   = None,
            MC_step_const = MC_STEP_CONST,
        )

        try:
            fit_result = fsd.fit(myres)
        except Exception as exc:
            log.debug("fsd.fit failed for %s: %s", csv_path.name, exc)
            return None
        if fit_result is None:
            return None

        output_params, conf_array, error, init, output_path = fit_result
        Q, Qc, w1, phi = output_params

        # Internal quality factor — same formula as DCMparams in
        # resonator.py:  1/Qi = 1/Q - Re(1/(Qc * e^{i*phi}))
        inv_Qi = Q**-1 - np.real((Qc * np.exp(1j * phi))**-1)
        if inv_Qi <= 0:          # unphysical fit -> reject
            return None
        Qi = 1.0 / inv_Qi

        # conf_array layout (from the repo): [Q, Qi, Qc, Qc_Re, phi, w1]
        Qi_err  = float(conf_array[1])
        Qc_err  = float(conf_array[2])
        phi_err = float(conf_array[4])
        fc_err  = float(conf_array[5]) / 1e9

        # Move the repo's PNG to sit next to the original CSV.
        if save_png:
            png_files = list(Path(output_path).glob("*.png"))
            if png_files:
                shutil.move(str(png_files[0]),
                            str(csv_path.with_suffix(".png")))

    return {
        "Qi":  Qi,        "Qi_err":  Qi_err,
        "Qc":  Qc,        "Qc_err":  Qc_err,
        "phi": phi,       "phi_err": phi_err,
        "fc":  w1 / 1e9,  "fc_err":  fc_err,
    }

In [ ]:
def process_power_map(
    power_map: Dict[str, List[Tuple]],
) -> Dict[str, pd.DataFrame]:
    """Fit every (timestamp, csv_path) pair.

    Returns
    -------
    dict : power_str -> DataFrame[timestamp, Qi, Qi_err, Qc, Qc_err,
                                  phi, phi_err, fc, fc_err]
    """
    results: Dict[str, pd.DataFrame] = {}

    for power, entries in sorted(power_map.items()):
        log.info("Processing power %s  (%d files)", power, len(entries))
        rows = []
        for ts, csv_path in entries:
            try:
                params = fit_one_file(csv_path, save_png=SAVE_CIRCLE_PLOTS)
            except Exception as exc:
                log.warning("Skipping %s — error: %s", csv_path.name, exc)
                continue
            if params is None:
                log.warning("Skipping %s — fit returned no result.",
                            csv_path.name)
                continue
            rows.append({"timestamp": ts, **params})
            plt.close("all")   # free figure memory inside the loop

        if rows:
            results[power] = (pd.DataFrame(rows)
                              .sort_values("timestamp")
                              .reset_index(drop=True))
            log.info("  -> %d successful fits", len(results[power]))
        else:
            log.warning("  -> No successful fits for power %s", power)

    return results


results = process_power_map(power_map)
if not results:
    raise RuntimeError("No fitting results were produced. "
                       "Check your data files and the log above.")

## Section 7 — Step 4b · Time-series plots and Welch PSDs

Two figures:

1. **Time sweeps** — 2 × 2 panel of Qᵢ, Qc, φ, f꜀ vs elapsed time, with ±1σ
   fit-uncertainty bands, all powers overlaid.
2. **PSDs** — log-log power spectral density of the Qᵢ and f꜀ fluctuations,
   computed with **Welch's method** (`scipy.signal.welch`, Hann window,
   averaged segments). The sampling rate is inferred from the median interval
   between sweep timestamps.

In [ ]:
def _power_colours(powers: List[str]) -> Dict[str, str]:
    """Fixed colour sequence: black, red, green, blue (cycles past 4)."""
    palette = ["black", "red", "green", "blue"]
    return {p: palette[i % len(palette)] for i, p in enumerate(sorted(powers))}


def _elapsed_seconds(df: pd.DataFrame) -> np.ndarray:
    """Elapsed time in seconds relative to the first fit in this group."""
    t0 = df["timestamp"].iloc[0]
    return (df["timestamp"] - t0).dt.total_seconds().to_numpy()


def _compute_psd(values: np.ndarray,
                 timestamps: pd.Series) -> Tuple[np.ndarray, np.ndarray]:
    """One-sided PSD of a (roughly) uniformly sampled series via Welch.

    The sampling rate is the inverse of the median interval between
    timestamps. Welch's method splits the series into overlapping
    Hann-windowed segments and averages their periodograms, trading
    frequency resolution for a much less noisy spectrum.

    Returns (frequencies_Hz, psd). Units: [values]^2 / Hz.
    """
    t_sec = timestamps.astype(np.int64).to_numpy() / 1e9   # ns -> s
    dt    = float(np.median(np.diff(t_sec)))
    fs    = 1.0 / dt if dt > 0 else 1.0

    # Segment length: at most 256 samples, at least the full series.
    nperseg = min(len(values), 256)
    freqs, psd = sig.welch(values, fs=fs, window="hann",
                           nperseg=nperseg, detrend="constant")
    return freqs, psd

In [ ]:
def plot_time_sweeps(results: Dict[str, pd.DataFrame],
                     title_prefix: str = "") -> plt.Figure:
    """2x2 grid of Qi, Qc, phi, fc vs elapsed time, one line per power."""
    params  = ["Qi", "Qc", "phi", "fc"]
    colours = _power_colours(list(results.keys()))

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for ax, param in zip(axes, params):
        label = PARAM_LABELS[param]
        for power, df in sorted(results.items()):
            t   = _elapsed_seconds(df)
            val = df[param].to_numpy()
            err = df[f"{param}_err"].to_numpy()
            c   = colours[power]
            ax.plot(t, val, color=c, linewidth=1.5, label=power)
            ax.fill_between(t, val - err, val + err,      # ±1σ band
                            color=c, alpha=0.20, linewidth=0)
        ax.set_xlabel("Elapsed time (s)", fontsize=11)
        ax.set_ylabel(label, fontsize=12)
        ax.set_title(label, fontsize=12)
        ax.grid(True, linestyle="--", alpha=0.4)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        axes[0].legend(handles, labels, title="Input power",
                       fontsize=9, title_fontsize=9, loc="best",
                       framealpha=0.7)

    fig.suptitle(f"{title_prefix} — Time-Sweep Resonator Parameters",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    return fig


fig_ts = plot_time_sweeps(results, title_prefix=ANALYSIS_DIR.name)
if SAVE_PLOTS:
    out_path = ANALYSIS_DIR / "time_sweep_parameters.png"
    fig_ts.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path}")
plt.show()

In [ ]:
def plot_psd(results: Dict[str, pd.DataFrame],
             title_prefix: str = "") -> plt.Figure:
    """Log-log Welch PSDs of Qi and fc noise, all powers overlaid."""
    colours = _power_colours(list(results.keys()))
    fig, (ax_qi, ax_fc) = plt.subplots(1, 2, figsize=(14, 5))

    for power, df in sorted(results.items()):
        if len(df) <= 4:          # too few points for a meaningful PSD
            continue
        c = colours[power]
        f, p = _compute_psd(df["Qi"].to_numpy(), df["timestamp"])
        ax_qi.loglog(f[1:], p[1:], color=c, linewidth=1.4, label=power)

        fc_hz = df["fc"].to_numpy() * 1e9          # GHz -> Hz
        f, p  = _compute_psd(fc_hz, df["timestamp"])
        ax_fc.loglog(f[1:], p[1:], color=c, linewidth=1.4, label=power)

    for ax, title, ylabel in [
        (ax_qi, r"PSD of $Q_i$", r"$S_{Q_i}\,\mathrm{[Hz^{-1}]}$"),
        (ax_fc, r"PSD of $f_c$", r"$S_{f_c}\,\mathrm{[Hz^2/Hz]}$"),
    ]:
        ax.set_xlabel("Frequency (Hz)", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.grid(True, which="both", linestyle="--", alpha=0.35)
        h, lb = ax.get_legend_handles_labels()
        if h:
            ax.legend(h, lb, title="Input power",
                      fontsize=9, title_fontsize=9, framealpha=0.7)

    fig.suptitle(f"{title_prefix} — Welch PSD of Parameter Noise",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    return fig


fig_psd = plot_psd(results, title_prefix=ANALYSIS_DIR.name)
if SAVE_PLOTS:
    out_path = ANALYSIS_DIR / "psd_noise.png"
    fig_psd.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path}")
plt.show()

## Section 8 — Save fitted parameters

One CSV per power level, written to `<ANALYSIS_DIR>/fitted_params/`.

In [ ]:
def save_results(results: Dict[str, pd.DataFrame], output_dir: Path) -> None:
    """Write one CSV per power level to <output_dir>/fitted_params/."""
    out = output_dir / "fitted_params"
    out.mkdir(exist_ok=True)
    for power, df in results.items():
        safe = re.sub(r"[\\/]", "_", power)   # sanitise for a filename
        path = out / f"{safe}.csv"
        df.to_csv(path, index=False)
        log.info("Saved -> %s", path)


if SAVE_CSVS:
    save_results(results, ANALYSIS_DIR)

first_power = sorted(results.keys())[0]
print(f"\nPreview — {first_power}:")
display(results[first_power].head())